In [ ]:
%pip install dlt[rest_api] requests

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.stage")

In [ ]:
import dlt
import requests
import pandas as pd
from delta.tables import DeltaTable

In [ ]:
# ECB Statistical Data Warehouse REST API — free, no API key
# Both series confirmed working (monthly frequency M, not business-day B)
SERIES = {
    "euribor_3m": ("FM", "M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA"),
    "euribor_6m": ("FM", "M.U2.EUR.RT.MM.EURIBOR6MD_.HSTA"),
}

BASE = "https://data-api.ecb.europa.eu/service/data"

In [ ]:
@dlt.resource(name="macro_rates", write_disposition="merge", primary_key=["date", "indicator"])
def fetch_ecb_rates():
    base = "https://data-api.ecb.europa.eu/service/data"
    for indicator_name, (dataset, series_key) in SERIES.items():
        url = f"{base}/{dataset}/{series_key}"
        r = requests.get(url, params={"format": "jsondata", "startPeriod": "2020-01-01"}, timeout=30)
        if r.status_code != 200:
            print(f"  SKIP {indicator_name}: {r.status_code} — {url}")
            continue
        payload = r.json()
        series_data = payload["dataSets"][0]["series"]
        series_idx = list(series_data.keys())[0]
        obs = series_data[series_idx]["observations"]
        periods = payload["structure"]["dimensions"]["observation"][0]["values"]
        for i, period in enumerate(periods):
            obs_val = obs.get(str(i))
            if obs_val and obs_val[0] is not None:
                yield {"date": period["id"], "indicator": indicator_name, "value": float(obs_val[0])}

records = list(fetch_ecb_rates())
print(f"Fetched {len(records)} ECB macro records")
if records:
    pd.DataFrame(records).groupby("indicator").size().reset_index(name="rows")

In [ ]:
spark_df = spark.createDataFrame(pd.DataFrame(records))

tbl = "stocks.stage.macro_rates_raw"
if not spark.catalog.tableExists(tbl):
    spark_df.write.mode("overwrite").format("delta").saveAsTable(tbl)

DeltaTable.forName(spark, tbl).alias("t").merge(
    spark_df.alias("s"),
    "s.date = t.date AND s.indicator = t.indicator"
).whenNotMatchedInsertAll().execute()

print(f"stocks.stage.macro_rates_raw: {spark.table(tbl).count()} rows")